In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

print("="*60)
print("Drug Multi-Class Classification - 5-Fold Cross Validation (SMILES-based grouping)")
print("80% data for 5-fold CV, 20% independent test set")
print("="*60)


# ==================== 1. Model Definition ====================
class SpectrumEncoder(nn.Module):
    def __init__(self, input_dim=561, hidden_dim=256):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc = nn.Sequential(
            nn.Linear(256, hidden_dim),
            nn.BatchNorm1d(hidden_dim)
        )
    
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        h = self.conv_block(x).squeeze(-1)
        embed = self.fc(h)
        return F.normalize(embed, p=2, dim=1)


class MultiClassClassifier(nn.Module):
    def __init__(self, encoder, num_classes=9, input_dim=256, freeze_encoder=True):
        super().__init__()
        self.encoder = encoder
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        embed = self.encoder(x)
        logits = self.classifier(embed)
        return logits


# ==================== 2. Data Parsing ====================
def parse_msp_robust(msp_file, min_peaks=5, min_intensity=10):
    """Robust MSP parsing: split by entry to avoid state leakage"""
    with open(msp_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    parts = content.split('Name:')
    compounds = []
    
    for part in parts[1:]:
        lines = part.strip().splitlines()
        if not lines:
            continue
        name = lines[0].strip()
        rest = '\n'.join(lines[1:])
        
        smiles = ''
        formula = ''
        mw = 0.0
        peaks = []
        
        # Extract fields
        for line in lines[1:]:
            if line.startswith('SMILES:') or line.startswith('Smiles:'):
                smiles = line.split(':', 1)[1].strip()
            elif line.startswith('Formula:'):
                formula = line.split(':', 1)[1].strip()
            elif line.startswith('MW:'):
                try:
                    mw = float(line.split(':', 1)[1].strip())
                except:
                    mw = 0.0
        
        # Parse peak data
        idx_num_peaks = rest.find('Num peaks:')
        if idx_num_peaks != -1:
            peak_part = rest[idx_num_peaks:]
            lines_peak = peak_part.splitlines()
            peak_data_str = ' '.join(lines_peak[1:])
            peak_data_str = peak_data_str.replace(';', ' ').replace('\n', ' ')
            tokens = peak_data_str.split()
            for i in range(0, len(tokens)-1, 2):
                try:
                    mz = float(tokens[i])
                    intensity = float(tokens[i+1])
                    if mz > 0 and intensity >= min_intensity:
                        peaks.append((mz, intensity))
                except ValueError:
                    continue
        
        if len(peaks) >= min_peaks:
            compounds.append({
                'name': name,
                'smiles': smiles,
                'formula': formula,
                'mw': mw,
                'peaks': peaks
            })
    
    return compounds


def peaks_to_vector(peaks, mz_min=40, mz_max=600):
    dim = mz_max - mz_min + 1
    vec = np.zeros(dim)
    for mz, intensity in peaks:
        if mz_min <= mz <= mz_max:
            idx = int(round(mz - mz_min))
            if 0 <= idx < dim:
                vec[idx] += intensity
    return vec


def preprocess_spectra(spectra):
    tic = spectra.sum(axis=1, keepdims=True)
    spectra = spectra / (tic + 1e-8)
    spectra = np.sqrt(spectra)
    return spectra


def load_database(database_path, category_name, min_peaks=5, min_intensity=10):
    """Load a single database"""
    if not database_path.exists():
        print(f"  Warning: File not found: {database_path}")
        return None, None
    
    compounds = parse_msp_robust(str(database_path), min_peaks, min_intensity)
    if len(compounds) == 0:
        print(f"  Warning: No valid compounds parsed: {database_path}")
        return None, None
    
    spectra = np.array([peaks_to_vector(c['peaks']) for c in compounds])
    spectra = preprocess_spectra(spectra)
    
    print(f"  OK {category_name}: {len(compounds)} compounds, {spectra.shape[1]} dims")
    return compounds, spectra


# ==================== 3. Grouped Cross-Validation Split ====================
def stratified_group_kfold(groups, labels, n_splits=5, random_state=42):
    np.random.seed(random_state)
    unique_groups = np.unique(groups)
    group_to_label = {}
    for g in unique_groups:
        mask = groups == g
        group_to_label[g] = labels[mask][0]
    
    label_to_groups = defaultdict(list)
    for g, label in group_to_label.items():
        label_to_groups[label].append(g)
    
    folds = {i: [] for i in range(n_splits)}
    for label, group_list in label_to_groups.items():
        shuffled = group_list.copy()
        np.random.shuffle(shuffled)
        for i, g in enumerate(shuffled):
            fold_idx = i % n_splits
            folds[fold_idx].append(g)
    
    fold_indices = []
    for val_fold in range(n_splits):
        val_groups = folds[val_fold]
        train_groups = []
        for f in range(n_splits):
            if f != val_fold:
                train_groups.extend(folds[f])
        fold_indices.append({
            'train_groups': train_groups,
            'val_groups': val_groups
        })
    return fold_indices


# ==================== 4. Training Function ====================
def train_fold(model, train_loader, val_loader, device, num_classes, epochs=1000, lr=0.001, patience=30):
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6
    )
    criterion = nn.CrossEntropyLoss()
    
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for batch_spec, batch_labels in train_loader:
            batch_spec = batch_spec.to(device)
            batch_labels = batch_labels.to(device).long()
            optimizer.zero_grad()
            logits = model(batch_spec)
            loss = criterion(logits, batch_labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch_spec.size(0)
            preds = logits.argmax(dim=1)
            train_correct += (preds == batch_labels).sum().item()
            train_total += batch_spec.size(0)
        
        avg_train_loss = train_loss / train_total
        train_acc = train_correct / train_total
        
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for batch_spec, batch_labels in val_loader:
                batch_spec = batch_spec.to(device)
                batch_labels = batch_labels.to(device).long()
                logits = model(batch_spec)
                loss = criterion(logits, batch_labels)
                val_loss += loss.item() * batch_spec.size(0)
                preds = logits.argmax(dim=1)
                val_correct += (preds == batch_labels).sum().item()
                val_total += batch_spec.size(0)
        
        avg_val_loss = val_loss / val_total
        val_acc = val_correct / val_total
        
        scheduler.step(avg_val_loss)
        if epoch % 10 == 0 or epoch == 1:
            print(f'  Epoch {epoch:4d} | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2%} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2%}')
        
        if avg_val_loss < best_val_loss - 1e-4:
            best_val_loss = avg_val_loss
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'  Early stopping at epoch {epoch}')
                break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model


def evaluate_model(model, data_loader, device):
    """Evaluate model and return predictions"""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    
    with torch.no_grad():
        for batch_spec, batch_labels in data_loader:
            batch_spec = batch_spec.to(device)
            logits = model(batch_spec)
            probs = F.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch_labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    
    try:
        n_classes = all_probs.shape[1]
        all_labels_bin = label_binarize(all_labels, classes=range(n_classes))
        auc_ovr = roc_auc_score(all_labels_bin, all_probs, average='macro', multi_class='ovr')
    except:
        auc_ovr = 0.0
    
    return {
        'accuracy': acc,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'auc_ovr': auc_ovr,
        'preds': all_preds,
        'labels': all_labels,
        'probs': all_probs,
    }


# ==================== 5. Main Program ====================
def main():
    # Configuration
    base_dir = Path(r"D:\DL\cann\建模")
    databases = {
        'Fentanyls': 'Fentanyls_database.msp',
        'Cathinones': 'Cathinones_database.msp',
        'Synthetic_Cannabinoids': 'Synthetic_Cannabinoids_database.msp',
        'Arylcyclohexylamines': 'Arylcyclohexylamines_database.msp',
        'Benzodiazepines': 'Benzodiazepines_database.msp',
        'Nitazenes': 'Nitazenes_database.msp',
        'Opiates': 'Opiates_database.msp',
        'Phenethylamines': 'Phenethylamines_database.msp',
        'Tryptamines': 'Tryptamines_database.msp',
    }
    class_names = list(databases.keys())
    num_classes = len(class_names)
    
    encoder_path = base_dir / "best_model_0725.pt"
    if not encoder_path.exists():
        encoder_path = base_dir / "pretrained_model_v2" / "pretrained_encoder_final.pt"
    output_dir = base_dir / "multi_class_cv"
    output_dir.mkdir(exist_ok=True)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nDevice: {device}")
    print(f"Pre-trained model: {encoder_path}")
    print(f"Output directory: {output_dir}")
    
    # ---- 1. Load databases ----
    print("\n" + "="*60)
    print("1. Loading Databases")
    print("="*60)
    all_compounds = {}
    all_spectra = {}
    for category, filename in databases.items():
        db_path = base_dir / filename
        compounds, spectra = load_database(db_path, category)
        if compounds is not None and spectra is not None:
            all_compounds[category] = compounds
            all_spectra[category] = spectra
        else:
            all_compounds[category] = []
            all_spectra[category] = np.array([])
            print(f"  Warning: {category} load failed, using empty list")
    
    print("\n" + "="*60)
    print("Database Load Summary")
    print("="*60)
    total_compounds, total_spectra = 0, 0
    print(f"\n{'Category':<30} {'Compounds':<12} {'Spectra':<12}")
    print("-"*60)
    for category in all_compounds.keys():
        compounds = all_compounds[category]
        spectra = all_spectra[category]
        total_compounds += len(compounds)
        total_spectra += len(spectra)
        print(f"{category:<30} {len(compounds):<12} {len(spectra):<12}")
    print("-"*60)
    print(f"{'Total':<30} {total_compounds:<12} {total_spectra:<12}")
    print("="*60)
    
    # ---- 2. Build dataset ----
    print("\n" + "="*60)
    print("2. Building Dataset (SMILES-based grouping)")
    print("="*60)
    X_list, y_list, group_list, name_list = [], [], [], []
    for class_idx, category in enumerate(class_names):
        compounds = all_compounds[category]
        spectra = all_spectra[category]
        for comp_idx, comp in enumerate(compounds):
            X_list.append(spectra[comp_idx])
            y_list.append(class_idx)
            group_list.append(len(group_list))
            name_list.append(comp['name'])
    
    X = np.array(X_list)
    y = np.array(y_list)
    groups = np.array(group_list)
    names = np.array(name_list)
    
    print(f"  Total samples: {len(X)}")
    print(f"  Total groups (compounds): {len(np.unique(groups))}")
    print(f"  Feature dimension: {X.shape[1]}")
    
    # ---- 3. Split 80% train + 20% test ----
    print("\n" + "="*60)
    print("3. Splitting 80% Train + 20% Test (SMILES-based grouping)")
    print("="*60)
    unique_groups = np.unique(groups)
    group_to_label = {g: y[groups == g][0] for g in unique_groups}
    label_to_groups = defaultdict(list)
    for g, label in group_to_label.items():
        label_to_groups[label].append(g)
    
    train_groups_all, test_groups_all = [], []
    for label, g_list in label_to_groups.items():
        shuffled = g_list.copy()
        np.random.seed(42)
        np.random.shuffle(shuffled)
        n_test = int(len(shuffled) * 0.2)
        test_groups_all.extend(shuffled[:n_test])
        train_groups_all.extend(shuffled[n_test:])
    
    train_mask = np.isin(groups, train_groups_all)
    test_mask = np.isin(groups, test_groups_all)
    X_train_all, y_train_all, names_train_all = X[train_mask], y[train_mask], names[train_mask]
    X_test, y_test, names_test = X[test_mask], y[test_mask], names[test_mask]
    train_groups_all_actual = groups[train_mask]
    
    print(f"\n  Train groups: {len(train_groups_all)} compounds, {len(X_train_all)} spectra")
    print(f"  Test groups: {len(test_groups_all)} compounds, {len(X_test)} spectra")
    overlap = set(train_groups_all) & set(test_groups_all)
    print(f"  Group overlap: {'OK (None)' if not overlap else f'Warning: {len(overlap)} overlaps'}")
    
    print("\n  Per-class distribution:")
    for i, name in enumerate(class_names):
        train_count = np.sum(y_train_all == i)
        test_count = np.sum(y_test == i)
        print(f"    {name:30s}: Train={train_count:4d}, Test={test_count:4d}")
    
    # ---- 4. 5-Fold Cross Validation ----
    print("\n" + "="*60)
    print("4. 5-Fold Cross Validation (on 80% train groups)")
    print("="*60)
    n_splits = 5
    fold_indices = stratified_group_kfold(train_groups_all_actual, y_train_all, n_splits=n_splits, random_state=42)
    for fold_idx, fold in enumerate(fold_indices):
        train_g_fold = fold['train_groups']
        val_g_fold = fold['val_groups']
        train_mask_fold = np.isin(train_groups_all_actual, train_g_fold)
        val_mask_fold = np.isin(train_groups_all_actual, val_g_fold)
        print(f"  Fold {fold_idx + 1}: Train groups={len(train_g_fold)}, spectra={np.sum(train_mask_fold)}; Val groups={len(val_g_fold)}, spectra={np.sum(val_mask_fold)}")
    
    # ---- 5. Load pre-trained encoder ----
    print("\n" + "="*60)
    print("5. Loading Pre-trained Encoder")
    print("="*60)
    base_encoder = SpectrumEncoder(input_dim=561, hidden_dim=256).to(device)
    if encoder_path.exists():
        checkpoint = torch.load(str(encoder_path), map_location=device)
        if isinstance(checkpoint, dict):
            if 'encoder_state_dict' in checkpoint:
                state_dict = checkpoint['encoder_state_dict']
            elif 'state_dict' in checkpoint:
                state_dict = checkpoint['state_dict']
            else:
                state_dict = checkpoint
        else:
            state_dict = checkpoint
        if any(k.startswith('module.') for k in state_dict.keys()):
            state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
        if any(k.startswith('encoder.') for k in state_dict.keys()):
            state_dict = {k.replace('encoder.', ''): v for k, v in state_dict.items()}
        base_encoder.load_state_dict(state_dict, strict=False)
        print(f"  Loaded: {encoder_path}")
    else:
        print(f"  Warning: Pre-trained model not found, using random initialization")
    
    # ---- 6. 5-Fold Cross Validation Training ----
    print("\n" + "="*60)
    print("6. 5-Fold Cross Validation Training")
    print("="*60)
    fold_results = []
    batch_size = 64
    
    for fold_idx, fold in enumerate(fold_indices):
        print(f"\n{'='*60}")
        print(f"Fold {fold_idx + 1}/{n_splits}")
        print(f"{'='*60}")
        
        train_g_fold = fold['train_groups']
        val_g_fold = fold['val_groups']
        train_mask_fold = np.isin(train_groups_all_actual, train_g_fold)
        val_mask_fold = np.isin(train_groups_all_actual, val_g_fold)
        
        X_train = X_train_all[train_mask_fold]
        y_train = y_train_all[train_mask_fold]
        names_train = names_train_all[train_mask_fold]
        X_val = X_train_all[val_mask_fold]
        y_val = y_train_all[val_mask_fold]
        names_val = names_train_all[val_mask_fold]
        
        train_dataset = TensorDataset(
            torch.tensor(X_train, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.long)
        )
        val_dataset = TensorDataset(
            torch.tensor(X_val, dtype=torch.float32),
            torch.tensor(y_val, dtype=torch.long)
        )
        test_dataset = TensorDataset(
            torch.tensor(X_test, dtype=torch.float32),
            torch.tensor(y_test, dtype=torch.long)
        )
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
        train_eval_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
        
        print(f"  Train: {len(X_train)} samples")
        print(f"  Val: {len(X_val)} samples")
        print(f"  Test: {len(X_test)} samples")
        
        encoder = SpectrumEncoder(input_dim=561, hidden_dim=256).to(device)
        encoder.load_state_dict(base_encoder.state_dict())
        model = MultiClassClassifier(encoder, num_classes=num_classes, freeze_encoder=True).to(device)
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Trainable params: {trainable_params:,}")
        
        model = train_fold(
            model, train_loader, val_loader, device, num_classes, 
            epochs=1000, lr=0.001, patience=30
        )
        
        train_results = evaluate_model(model, train_eval_loader, device)
        val_results = evaluate_model(model, val_loader, device)
        test_results = evaluate_model(model, test_loader, device)
        
        fold_results.append({
            'fold': fold_idx + 1,
            'training': train_results,
            'validation': val_results,
            'test': test_results
        })
        
        print(f"\n  Fold {fold_idx + 1} Results:")
        print(f"    Train Acc: {train_results['accuracy']:.2%}")
        print(f"    Val Acc: {val_results['accuracy']:.2%}")
        print(f"    Test Acc: {test_results['accuracy']:.2%}")
        print(f"    Test Macro F1: {test_results['macro_f1']:.4f}")
        print(f"    Test AUC: {test_results['auc_ovr']:.4f}")
        
        # Save each fold model
        fold_model_path = output_dir / f'fold_{fold_idx + 1}_model.pt'
        torch.save({
            'encoder_state_dict': encoder.state_dict(),
            'classifier_state_dict': model.classifier.state_dict(),
        }, str(fold_model_path))
        print(f"  Model saved: {fold_model_path}")
    
    # ---- 7. Summary Results ----
    print("\n" + "="*60)
    print("7. 5-Fold Cross Validation Summary (Test Set)")
    print("="*60)
    test_accs = [r['test']['accuracy'] for r in fold_results]
    test_f1s = [r['test']['macro_f1'] for r in fold_results]
    test_aucs = [r['test']['auc_ovr'] for r in fold_results]
    
    avg_acc = np.mean(test_accs)
    std_acc = np.std(test_accs)
    avg_f1 = np.mean(test_f1s)
    std_f1 = np.std(test_f1s)
    avg_auc = np.mean(test_aucs)
    std_auc = np.std(test_aucs)
    
    print(f"\n  5-Fold CV Test Results:")
    print(f"    Avg Accuracy: {avg_acc:.2%} ± {std_acc:.2%}")
    print(f"    Avg Macro F1: {avg_f1:.4f} ± {std_f1:.4f}")
    print(f"    Avg AUC: {avg_auc:.4f} ± {std_auc:.4f}")
    
    print(f"\n  Per-fold Test Results:")
    for i, result in enumerate(fold_results):
        print(f"    Fold {i+1}: Acc={result['test']['accuracy']:.2%}, Macro F1={result['test']['macro_f1']:.4f}, AUC={result['test']['auc_ovr']:.4f}")
    
    # ---- 8. Save Best Model ----
    print("\n" + "="*60)
    print("8. Saving Best Model")
    print("="*60)
    
    best_fold_idx = np.argmax(test_accs)
    best_fold = fold_results[best_fold_idx]
    print(f"  Best Fold: Fold {best_fold_idx + 1}")
    print(f"    Test Accuracy: {test_accs[best_fold_idx]:.2%}")
    print(f"    Test Macro F1: {test_f1s[best_fold_idx]:.4f}")
    print(f"    Test AUC: {test_aucs[best_fold_idx]:.4f}")
    
    best_model_path = output_dir / f'fold_{best_fold_idx + 1}_model.pt'
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    final_model_path = output_dir / f'best_multi_class_model_{timestamp}.pt'
    
    best_encoder = SpectrumEncoder(input_dim=561, hidden_dim=256).to(device)
    best_model = MultiClassClassifier(best_encoder, num_classes=num_classes, freeze_encoder=True).to(device)
    
    if best_model_path.exists():
        checkpoint = torch.load(str(best_model_path), map_location=device)
        best_encoder.load_state_dict(checkpoint['encoder_state_dict'])
        best_model.classifier.load_state_dict(checkpoint['classifier_state_dict'])
        print(f"  Loaded best fold model weights: {best_model_path}")
    else:
        print(f"  Warning: Best fold model file not found: {best_model_path}")
    
    torch.save({
        'model_state_dict': {
            'encoder_state_dict': best_encoder.state_dict(),
            'classifier_state_dict': best_model.classifier.state_dict(),
        },
        'config': {
            'num_classes': num_classes,
            'class_names': class_names,
            'input_dim': 561,
            'hidden_dim': 256,
            'best_fold': best_fold_idx + 1,
            'test_accuracy': test_accs[best_fold_idx],
            'test_macro_f1': test_f1s[best_fold_idx],
            'test_auc': test_aucs[best_fold_idx],
            'avg_accuracy': avg_acc,
            'avg_macro_f1': avg_f1,
            'avg_auc': avg_auc,
            'n_splits': n_splits,
        },
        'timestamp': timestamp,
    }, str(final_model_path))
    print(f"  Best model saved: {final_model_path}")
    
    lightweight_model_path = output_dir / f'best_multi_class_model_weights_{timestamp}.pt'
    torch.save({
        'encoder_state_dict': best_encoder.state_dict(),
        'classifier_state_dict': best_model.classifier.state_dict(),
    }, str(lightweight_model_path))
    print(f"  Lightweight model weights saved: {lightweight_model_path}")
    
    latest_model_path = output_dir / 'best_multi_class_model_latest.pt'
    torch.save({
        'model_state_dict': {
            'encoder_state_dict': best_encoder.state_dict(),
            'classifier_state_dict': best_model.classifier.state_dict(),
        },
        'config': {
            'num_classes': num_classes,
            'class_names': class_names,
            'input_dim': 561,
            'hidden_dim': 256,
            'best_fold': best_fold_idx + 1,
            'test_accuracy': test_accs[best_fold_idx],
            'test_macro_f1': test_f1s[best_fold_idx],
            'test_auc': test_aucs[best_fold_idx],
            'avg_accuracy': avg_acc,
            'avg_macro_f1': avg_f1,
            'avg_auc': avg_auc,
            'n_splits': n_splits,
        },
        'timestamp': timestamp,
    }, str(latest_model_path))
    print(f"  Latest model saved: {latest_model_path}")
    
    print(f"\n{'='*60}")
    print("Complete!")
    print(f"{'='*60}")
    print(f"  Output directory: {output_dir}")
    print(f"  Best model: {final_model_path}")
    print(f"  Latest model: {latest_model_path}")


if __name__ == "__main__":
    main()